# Simple Adaptive walkthrough — mock agents or Groq
Run the six steps in order using this project's `.venv` Python kernel.

This is the shorter version of `adaptive_walkthrough.ipynb`. Both use the same single-pole test.

We use the Plant Agent-format `single_pole.json` and its only test: initial state `[0.05, 0.0]`, reference `theta = 0.0`. Angles are in radians; zero is upright. The test includes small uncertainty and disturbance terms. Editable experiment inputs mirror the Adaptive Streamlit controls: plant/test conditions, output reference, simulation time/step, tuning enabled, target error, tuning rounds, priorities, and model selection. Uncertainty/disturbance entries represent answers normally confirmed through the Clarifier chat.

The confirmed artifact replaces that chat in this example. Clarifier calls and their costs are not included. There is no numbered scenario-level setting.


## 1. Set up imports and find the project
No API calls are made here. Start the kernel inside the repository or its parent folder.


In [ ]:
import copy
import json
import os
import sys
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv

starts = (Path.cwd(), *Path.cwd().parents)
candidates = [p for start in starts for p in (start, start / "LabCD_NewModules-master")]
REPO_ROOT = next(
    (p for p in candidates if (p / "backend_core" / "AgentAdaptive").is_dir()
     and (p / "benchmarks" / "Adaptive" / "artifacts").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Start the kernel inside the repository or its parent folder.")
for directory in (REPO_ROOT, REPO_ROOT / "packages" / "labcd_agents" / "src"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
ADAPTIVE_DIR = REPO_ROOT / "benchmarks" / "Adaptive"
print("Project:", REPO_ROOT)


## 2. Choose site-level inputs
`ARTIFACT_FILE` and `TEST_ID` select a saved plant and saved IC/reference case; they are notebook conveniences, not extra site controls.

Tuning is enabled for this example (the site's checkbox initially defaults to off).
`TARGET_ERROR_FRACTION` uses the site's default 0.02; its allowed range is 0.005–0.10 in steps of 0.005. The backend evaluates it against the task scale, despite the site's "reference amplitude" label.

The example chooses "Control effort" as its only optional priority, so the Tuner can improve effort even when initial tracking is already adequate. Use `None` for default priorities; tuning may then be skipped when initial tracking meets the target.

`None` for IC/reference/effect inputs below means use the selected artifact/test. The test already contains mild uncertainty and disturbance, so Adaptive estimators are active; supply an empty list for an effect to turn that effect off.

`USE_MOCK_AGENTS` and `MOCK_SEED` are local test controls. Mock gains vary reproducibly; performance improvements are not guaranteed. Mock usage is marked `simulated`, with a synthetic $0.001/token cost and zero actual API cost. The confirmed artifact skips Clarifier chat in both modes.


In [ ]:
ARTIFACT_FILE = "single_pole.json"
TEST_ID = "test_1"
USE_MOCK_AGENTS = True  # False uses real Groq agents.
MOCK_SEED = 42          # Repeatable mock proposals and synthetic token counts.

ENABLE_TUNING = True
TARGET_ERROR_FRACTION = 0.02
MAX_TUNING_ROUNDS = 4
TUNING_OBJECTIVES = {"control_effort": 3}  # Or None; importance values are 1–5.

SIMULATION_TIME = 10.0  # Adaptive form: 5, 10, 30, or 100 seconds.
SOLVER_STEP = 0.001    # Adaptive form: 0.0001, 0.001, or 0.01 seconds.
INITIAL_STATE = None  # Or a full state list, e.g. [0.05, 0.0].
REFERENCES = None     # Or an output/expression map, e.g. {"theta": "0"}.

UNCERTAINTY = None  # Or [], or [{"state": "omega", "expr": "-0.02 * omega"}].
DISTURBANCE = None  # Or [], or [{"state": "omega", "expr": "0.01 * sin(0.8*t)"}].

DESIGN_MODEL = "openai/gpt-oss-120b"
TUNER_MODEL = None  # None inherits the Design model, like the site's model picker.


## 3. Read the artifact and build the confirmed specification
Only the controller's native specification is passed forward. The artifact and test are not modified.

The test saves one reference expression per declared output; there is no target value for an untracked state. Test conditions are defined explicitly; the Adaptive runner has no numbered scenario-level preset.

The selected test's nonempty uncertainty/disturbance lists activate Adaptive estimation. Enabling LLM tuning alone does not activate estimators; the actual uncertainty or disturbance terms do.


In [ ]:
from backend_core.AgentAdaptive.tools import system_spec as spec_tools
from backend_core.AgentAdaptive.controller.structure_build import _build_structure_from_spec

artifact_path = ADAPTIVE_DIR / "artifacts" / ARTIFACT_FILE
artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
tests = artifact.get("tests", {}).get("combinations", [])
matching = [test for test in tests if test["test_id"] == TEST_ID]
if len(matching) != 1:
    raise ValueError(f"Choose one of these test IDs: {[test['test_id'] for test in tests]}")
test = copy.deepcopy(matching[0])

# Convert the Plant Agent output without adding fields to the artifact.
from backend_core.plant_compiler import PlantCompiler

compiler = PlantCompiler()
validation = compiler.validate(artifact)
if not validation.ok:
    raise ValueError(f"Invalid Plant Agent artifact: {validation.errors}")
applied_spec = compiler.generate_adaptive_spec(
    plant_output=artifact,
    pre_launch={
        "initial_state": test["initial_condition"],
        "total_simulation_time": SIMULATION_TIME,
        "solver_sample_time": SOLVER_STEP,
    },
)
dyn = applied_spec["dynamics"]
dyn["uncertainty"] = copy.deepcopy(test["uncertainty"])
dyn["disturbance"] = copy.deepcopy(test["disturbance"])
states = dyn["states"]
initial_values = np.asarray(test["initial_condition"] if INITIAL_STATE is None else INITIAL_STATE, dtype=float)
if initial_values.shape != (len(states),) or not np.all(np.isfinite(initial_values)):
    raise ValueError("IC must have one finite value per state.")
dyn["x0"] = initial_values.tolist()
saved_references = test["references"]
if (not isinstance(saved_references, list) or len(saved_references) != len(dyn["outputs"])
        or any(not isinstance(ref, dict) or not isinstance(ref.get("expr"), str)
               or not ref["expr"].strip() for ref in saved_references)):
    raise ValueError("The test needs one nonempty reference per declared output.")
saved_reference_map = {ref.get("output"): ref["expr"] for ref in saved_references}
if len(saved_reference_map) != len(saved_references) or set(saved_reference_map) != set(dyn["outputs"]):
    raise ValueError("Test references must name each declared output exactly once.")
reference_map = saved_reference_map if REFERENCES is None else REFERENCES
if set(reference_map) != set(dyn["outputs"]):
    raise ValueError("Provide one reference expression for each declared output.")
dyn["references"] = [{"output": name, "expr": str(reference_map[name])} for name in dyn["outputs"]]
dyn["sim_time"] = SIMULATION_TIME
dyn["solver_step"] = SOLVER_STEP
if UNCERTAINTY is not None:
    dyn["uncertainty"] = copy.deepcopy(UNCERTAINTY)
if DISTURBANCE is not None:
    dyn["disturbance"] = copy.deepcopy(DISTURBANCE)

if SIMULATION_TIME not in (5.0, 10.0, 30.0, 100.0) or SOLVER_STEP not in (0.0001, 0.001, 0.01):
    raise ValueError("Use the simulation time and solver-step choices offered by the Adaptive form.")
fraction_step = TARGET_ERROR_FRACTION / 0.005
if not (0.005 <= TARGET_ERROR_FRACTION <= 0.10) or not np.isclose(fraction_step, round(fraction_step)):
    raise ValueError("Target error must be 0.005–0.10, in steps of 0.005.")
if isinstance(MAX_TUNING_ROUNDS, bool) or not isinstance(MAX_TUNING_ROUNDS, int) or not 1 <= MAX_TUNING_ROUNDS <= 10:
    raise ValueError("Max tuning rounds must be an integer from 1 to 10.")
if TUNING_OBJECTIVES is not None:
    from backend_core.AgentAdaptive.tools.tuning_objectives import OBJECTIVES
    if any(key not in OBJECTIVES or isinstance(weight, bool) or not isinstance(weight, int)
           or not 1 <= weight <= 5 for key, weight in TUNING_OBJECTIVES.items()):
        raise ValueError("Use supported tuning priorities with integer importance values 1–5.")
gaps = spec_tools.missing_items(applied_spec)
if gaps or not np.all(np.isfinite(np.asarray(dyn["x0"], dtype=float))):
    raise ValueError(f"Incomplete or nonfinite specification: {gaps}")
_, limitations = _build_structure_from_spec(spec_tools.substitute_parameters(applied_spec))
if limitations:
    raise ValueError(f"Unsupported reference; choose a constant, sine, or cosine: {limitations}")

print("Plant:", applied_spec["system_name"], "| Test:", TEST_ID)
print("Initial state:", dyn["x0"])
print("References:", dyn["references"])
print("Adaptive estimators active:", bool(dyn["uncertainty"] or dyn["disturbance"]))


## 4. Run the Designer and optional Tuner
`USE_MOCK_AGENTS=True` uses seeded mock Designer, Tuner and Report Writer responses. Simulations and metrics are real. No API key is required. Set it to `False` for Groq API calls.
For real Groq runs, set `GROQ_API_KEY` in the repository-root `.env` or environment. No key is printed or saved.

Only user-level tuning arguments and the generated specification are supplied. Progress callbacks, cancellation callbacks, clarification records/usage, and simulation overrides use backend defaults. Timing comes from the specification.

Reporter inherits the selected Design model. Provider/key routing is temporary internal plumbing. No token-limit, temperature, or controller-gain overrides are made; the backend/environment supplies those defaults.

The import selects the implementation: `tuner_agent.py` uses mock agents; `tuner_agent_main.py` uses the original LLM agents. Real agent files keep the `_main` suffix.


In [ ]:
routing = {"LABCD_ADAPTIVE_SHOW_PLOTS": "0"}
if not USE_MOCK_AGENTS:
    load_dotenv(dotenv_path=REPO_ROOT / ".env", override=False)
    groq_key = os.getenv("GROQ_API_KEY")
    if not groq_key:
        raise RuntimeError("Set GROQ_API_KEY in the environment or the repository-root .env.")
    for role in ("design", "tuner", "clarifier", "reporter"):
        suffix = "" if role == "design" else "_" + role.upper()
        routing["LLM_PROVIDER" + suffix] = "groq"
        routing["OPENAI_API_KEY" + suffix] = groq_key
        routing["OPENAI_MODEL" + suffix] = (
            (TUNER_MODEL or DESIGN_MODEL) if role == "tuner" else DESIGN_MODEL
        )
    del groq_key
previous = {key: os.environ.get(key) for key in routing}
result = usage = tuning_history = best = None
try:
    os.environ.update(routing)
    if USE_MOCK_AGENTS:
        from backend_core.AgentAdaptive.agents.tuner_agent import run_full_pipeline
        agent_options = {"mock_seed": MOCK_SEED}
    else:
        from backend_core.AgentAdaptive.agents.tuner_agent_main import run_full_pipeline
        agent_options = {}

    result, usage, tuning_history, best = run_full_pipeline(
        description="",
        enable_tuning=ENABLE_TUNING,
        target_rms_frac=TARGET_ERROR_FRACTION,
        max_tuning_rounds=MAX_TUNING_ROUNDS,
        tuning_objectives=TUNING_OBJECTIVES,
        system_spec=applied_spec,
        **agent_options,
    )
finally:
    for key, value in previous.items():
        if value is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = value
    del routing, previous

print("Best parameters:")
pprint((best or {}).get("tuning"))
print("Final simulation metrics:")
pprint(result.get("final_metrics"))
print("Token totals:")
pprint(usage.get("total"))


## 5. Save a compact JSON report
Trajectories remain in memory for plotting but are not saved in this JSON.
The report records the applied specification, tuning inputs, best parameters/metrics, native final metrics, tuning history, and token usage.
Each run updates `batch_tuning_results.json` and `batch_tuning_results_brief.json` in the selected model folder. A new run for that model replaces the previous pair. Missing/nonfinite metrics become JSON `null`, not a fake success.

Mock reports go under `Adaptive/json_reports/results/mock/` and include the mock messages. Actual verification metrics come from the final controller simulation.


In [ ]:
def json_safe(value):
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if value is None or isinstance(value, (str, int, bool)):
        return value
    raise TypeError(f"Unsupported report value: {type(value).__name__}")

if result is None:
    raise RuntimeError("Run step 4 successfully before saving a report.")
summary = getattr((result.get("messages") or [None])[-1], "content", "")
report = {
    "artifact": ARTIFACT_FILE,
    "plant": applied_spec["system_name"],
    "test_id": TEST_ID,
    "applied_system_spec": applied_spec,
    "agent_mode": "mock" if USE_MOCK_AGENTS else "groq",
    "mock_seed": MOCK_SEED if USE_MOCK_AGENTS else None,
    "tuning_inputs": {
        "enabled": ENABLE_TUNING,
        "target_error_fraction": TARGET_ERROR_FRACTION,
        "max_rounds": MAX_TUNING_ROUNDS,
        "objectives": TUNING_OBJECTIVES,
    },
    "tuning": {
        "mock_agent_messages": result.get("mock_agent_messages", []),
        "best_params": (best or {}).get("tuning"),
        "best_metrics": (best or {}).get("metrics"),
        "best_round": (best or {}).get("round"),
        "history": tuning_history,
        "token_usage": usage,
    },
    "verification": result.get("final_metrics"),
    "pipeline_completed": result.get("final_metrics") is not None and bool(result.get("series")),
    "summary": summary,
}
output_dir = ADAPTIVE_DIR / "json_reports" / "results" / (
    "mock" if USE_MOCK_AGENTS else "groq-simple"
)
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "batch_tuning_results.json"
output_file.write_text(
    json.dumps(json_safe(report), indent=2, ensure_ascii=False, allow_nan=False) + "\n",
    encoding="utf-8",
)
print("Saved JSON:", output_file)

from benchmarks.Adaptive.result_brief import write_brief_result

brief_output_file = write_brief_result(output_file)
print("Saved brief JSON:", brief_output_file)
if not report["pipeline_completed"]:
    raise RuntimeError("Design/simulation did not complete. The diagnostic report was saved. " + str(summary))


## 6. Plot the final response
These curves use the backend's exported final simulation series, which may be downsampled for display. Metrics use the original simulation resolution.

Pipeline completion does not guarantee `verification.success=True`. That native verdict checks numerical health, boundedness, and tracking error; it is not the MPC exponential success score.

The original walkthrough remains available for detailed progress callbacks, report reloading, and additional configuration.


In [ ]:
series = result.get("series")
if not series:
    raise RuntimeError("No final simulation series is available; inspect the saved diagnostic.")
channels = series["channels"]
times = np.asarray(channels["t"]["data"], dtype=float)
actual = np.asarray(channels["y"]["data"], dtype=float)
reference = np.asarray(channels["ref"]["data"], dtype=float)
inputs = np.asarray(channels["u"]["data"], dtype=float)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
for i, name in enumerate(channels["y"]["names"]):
    axes[0].plot(times, actual[:, i], label=f"{name}: actual")
    axes[0].plot(times, reference[:, i], "--", label=f"{name}: reference")
for i, name in enumerate(channels["u"]["names"]):
    axes[1].plot(times, inputs[:, i], label=name)
axes[0].set_title(f"{applied_spec['system_name']} | {TEST_ID}")
axes[0].set_ylabel("Tracked output")
axes[1].set_ylabel("Control input")
axes[1].set_xlabel("Time (s)")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.show()
